# Revisar los resultados
Objetivo: ver si los resultados cambian cuando cambiamos alguna decisión (cómo medimos la distancia, dónde empieza la cuadrícula, cuántos puntos críticos tomamos, ordenar por muertos y el periodo) y comparar nuestros datos con el anuario oficial.
Todo es con víctimas de 2019 a 2025, igual que en el notebook 06.

In [1]:
import pandas as pd
import numpy as np

victimas = pd.read_csv("../data/processed/victimas_camaras.csv.gz")
siniestros = pd.read_csv("../data/processed/siniestros_camaras.csv.gz")
camaras = pd.read_csv("../data/processed/camaras.csv")

periodo = victimas[victimas["AA_Acc"] >= 2019].copy()
periodo["muerto"] = periodo["Gravedad_Indicador_30d"] == "MUERTO"

# los mismos cuadros de 300 m del notebook 06
lado = 0.0027
periodo["fila"] = (periodo["Latitud"] / lado).round().astype(int)
periodo["columna"] = (periodo["Longitud"] / lado).round().astype(int)
periodo["cuadro"] = periodo["fila"].astype(str) + "_" + periodo["columna"].astype(str)
print(periodo.shape)

(138178, 21)


In [2]:
# cuenta víctimas y muertos por cuadro y los ordena igual que en el notebook 06
def armar_cuadros(tabla, columna_cuadro):
    cuadros = tabla.groupby(columna_cuadro).agg(
        victimas=("Codigo_Accidentado", "count"),
        muertos=("muerto", "sum"),
        distancia_min=("distancia_m", "min")
    ).reset_index()
    cuadros["tiene_camara"] = cuadros["distancia_min"] <= 300
    cuadros = cuadros.sort_values(["victimas", "muertos", columna_cuadro], ascending=[False, False, True]).reset_index(drop=True)
    return cuadros

cuadros = armar_cuadros(periodo, "cuadro")
top50 = cuadros.head(50)
print("Puntos críticos sin cámara:", (~top50["tiene_camara"]).sum(), "de 50")

Puntos críticos sin cámara: 34 de 50


Cuántas víctimas quedaron después de limpiar

Contamos todas las víctimas antes de limpiar (como en el notebook 03) y las comparamos con las que quedaron.

In [3]:
carpeta = "../data/raw/Siniestros_viales/"

# el año de cada siniestro, también de los que quitamos al limpiar
anios = pd.read_csv(carpeta + "SIGAT_ANUARIO_2024_Siniestros.csv.gz", usecols=["Codigo_Accidente", "AA_Acc"])
anios_2025 = pd.read_excel(carpeta + "SIGAT_ANUARIO_2025.xlsx", sheet_name="Siniestros", usecols=["Codigo_Siniestro", "AA_Siniestro"])
anios_2025 = anios_2025.rename(columns={"Codigo_Siniestro": "Codigo_Accidente", "AA_Siniestro": "AA_Acc"})
anios = pd.concat([anios, anios_2025], ignore_index=True).drop_duplicates(subset="Codigo_Accidente", keep="last")

# todas las víctimas antes de limpiar
actores = pd.read_csv(carpeta + "SIGAT_ANUARIO_2024_Actor_vial.csv.gz", usecols=["Codigo_Accidentado", "Codigo_Accidente", "Gravedad_Indicador_30d"])
act_2025 = pd.read_excel(carpeta + "SIGAT_ANUARIO_2025.xlsx", sheet_name="Actor vial", usecols=["Codigo_Accidentado", "Codigo_Siniestro", "Gravedad_Indicador_30d"])
act_2025 = act_2025.rename(columns={"Codigo_Siniestro": "Codigo_Accidente"})
todas = pd.concat([actores, act_2025], ignore_index=True).drop_duplicates(subset="Codigo_Accidentado", keep="last")
todas = todas[todas["Gravedad_Indicador_30d"].isin(["HERIDO", "MUERTO"])]
todas = todas.merge(anios, on="Codigo_Accidente", how="left", validate="m:1")
print("Víctimas antes de limpiar:", len(todas), "| sin año:", todas["AA_Acc"].isna().sum())

antes = (todas["AA_Acc"] <= 2018).sum()
despues = (todas["AA_Acc"] >= 2019).sum()
quedan_antes = (victimas["AA_Acc"] <= 2018).sum()
quedan_despues = (victimas["AA_Acc"] >= 2019).sum()
print("2019-2025: quedaron", quedan_despues, "de", despues, "->", round(quedan_despues / despues * 100, 1), "%")
print("2015-2018: quedaron", quedan_antes, "de", antes, "-> se perdió el", round(100 - quedan_antes / antes * 100, 1), "%")

Víctimas antes de limpiar: 205182 | sin año: 0
2019-2025: quedaron 138178 de 140003 -> 98.7 %
2015-2018: quedaron 59852 de 65179 -> se perdió el 8.2 %


Comparar 2024 con el anuario oficial

El Anuario de Siniestralidad Vial 2024 de la Secretaría de Movilidad dice 565 muertos y 22.593 heridos.

In [4]:
del_2024 = victimas[victimas["AA_Acc"] == 2024]
print(del_2024["Gravedad_Indicador_30d"].value_counts())

Gravedad_Indicador_30d
HERIDO    22589
MUERTO      565
Name: count, dtype: int64


Medir la distancia con la fórmula exacta

En el notebook 05 pasamos los grados a metros (1 grado = 111.000 m) y usamos Pitágoras. Como la Tierra es casi una esfera, aquí usamos la fórmula del haversine, que mide la distancia sobre una esfera, para ver cuántas víctimas cambian de grupo (cerca o lejos de una cámara).

In [5]:
radio_tierra = 6371000
cams = camaras[["id_camara", "latitud", "longitud"]]
partes = []

for anio in range(2019, 2026):
    pares = siniestros[siniestros["AA_Acc"] == anio][["Codigo_Accidente", "Latitud", "Longitud"]].merge(cams, how="cross")
    lat1 = np.radians(pares["Latitud"])
    lat2 = np.radians(pares["latitud"])
    dlat = lat2 - lat1
    dlon = np.radians(pares["longitud"] - pares["Longitud"])
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    pares["distancia_exacta"] = 2 * radio_tierra * np.arcsin(np.sqrt(a))
    partes.append(pares.groupby("Codigo_Accidente").agg(distancia_exacta=("distancia_exacta", "min")).reset_index())

exacta = pd.concat(partes, ignore_index=True)
comparar = periodo.merge(exacta, on="Codigo_Accidente", how="left", validate="m:1")

cambian = (comparar["distancia_m"] <= 300) != (comparar["distancia_exacta"] <= 300)
print("Víctimas que cambian de grupo:", cambian.sum(), "de", len(comparar))
print("A 300 m o menos con la fórmula exacta:", round((comparar["distancia_exacta"] <= 300).mean() * 100, 1), "%")

Víctimas que cambian de grupo: 20 de 138178
A 300 m o menos con la fórmula exacta: 8.8 %


Correr la cuadrícula medio cuadro

Los cuadros dependen de dónde empieza la cuadrícula. La corremos medio cuadro hacia arriba, hacia un lado y en diagonal, y volvemos a contar los puntos críticos sin cámara.

In [6]:
for mover_fila, mover_columna in [(0.5, 0), (0, 0.5), (0.5, 0.5)]:
    fila2 = (periodo["Latitud"] / lado + mover_fila).round().astype(int)
    columna2 = (periodo["Longitud"] / lado + mover_columna).round().astype(int)
    periodo["cuadro_corrido"] = fila2.astype(str) + "_" + columna2.astype(str)
    otros = armar_cuadros(periodo, "cuadro_corrido")
    print("Cuadrícula corrida", (mover_fila, mover_columna), "->", (~otros.head(50)["tiene_camara"]).sum(), "de 50 sin cámara")

Cuadrícula corrida (0.5, 0) -> 29 de 50 sin cámara
Cuadrícula corrida (0, 0.5) -> 33 de 50 sin cámara
Cuadrícula corrida (0.5, 0.5) -> 29 de 50 sin cámara


Cambiar el número de puntos críticos

Usamos 50. Aquí miramos los 10, 20 y 100 primeros.

In [7]:
for n in [10, 20, 50, 100]:
    cuantos = (~cuadros.head(n)["tiene_camara"]).sum()
    print("Top", n, "->", cuantos, "sin cámara,", round(cuantos / n * 100), "%")

Top 10 -> 7 sin cámara, 70 %
Top 20 -> 12 sin cámara, 60 %
Top 50 -> 34 sin cámara, 68 %
Top 100 -> 72 sin cámara, 72 %


Ordenar por muertos en vez de víctimas

In [8]:
por_muertos = cuadros.sort_values(["muertos", "victimas", "cuadro"], ascending=[False, False, True]).head(50)
print("Muertos en el puesto 50:", por_muertos["muertos"].iloc[49])
print("Sin cámara en los 50 cuadros con más muertos:", (~por_muertos["tiene_camara"]).sum(), "de 50")

Muertos en el puesto 50: 7
Sin cámara en los 50 cuadros con más muertos: 41 de 50


Comparar 2019-2022 con 2023-2025

Armamos los 50 puntos críticos de cada periodo por separado y miramos cuáles están en las dos listas.

In [9]:
primeros = armar_cuadros(periodo[periodo["AA_Acc"] <= 2022], "cuadro").head(50)
ultimos = armar_cuadros(periodo[periodo["AA_Acc"] >= 2023], "cuadro").head(50)
en_los_dos = set(primeros["cuadro"]) & set(ultimos["cuadro"])
print("Puntos críticos que están en las dos listas:", len(en_los_dos), "de 50")

sin_camara = top50[~top50["tiene_camara"]]
se_repiten = sin_camara[sin_camara["cuadro"].isin(en_los_dos)]
print("De los", len(sin_camara), "sin cámara, están en las dos listas:", len(se_repiten))
print("Víctimas y muertos 2019-2025 en esos puntos:", se_repiten["victimas"].sum(), "y", se_repiten["muertos"].sum())

Puntos críticos que están en las dos listas: 24 de 50
De los 34 sin cámara, están en las dos listas: 16
Víctimas y muertos 2019-2025 en esos puntos: 3746 y 106


Cuántos cuadros tienen cámara y cuántas víctimas hay en ellos

In [10]:
pct_cuadros = cuadros["tiene_camara"].mean() * 100
pct_victimas = cuadros[cuadros["tiene_camara"]]["victimas"].sum() / cuadros["victimas"].sum() * 100
print("Cuadros con víctimas que tienen cámara:", round(pct_cuadros, 1), "%")
print("Víctimas en esos cuadros:", round(pct_victimas, 1), "%")

Cuadros con víctimas que tienen cámara: 7.7 %
Víctimas en esos cuadros: 13.9 %


## Resultados
- Quedó el 98,7 % de las víctimas de 2019-2025 (138.178 de 140.003). En 2015-2018 se perdió el 8,2 %, casi todo porque los siniestros no tenían coordenadas.
- En 2024 tenemos 565 muertos, igual que el anuario, y 22.589 heridos (el anuario dice 22.593).
- Con la fórmula exacta solo 20 de 138.178 víctimas cambian de grupo, y sigue dando 8,8 % cerca de una cámara.
- Corriendo la cuadrícula quedan entre 29 y 33 puntos críticos sin cámara (con la cuadrícula normal son 34).
- En los 10, 20, 50 y 100 primeros puntos, entre el 60 % y el 72 % no tiene cámara.
- Si ordenamos por muertos, 41 de los 50 cuadros con más muertos no tienen cámara.
- 24 de los 50 puntos críticos están en las dos listas (2019-2022 y 2023-2025). De los 34 sin cámara se repiten 16, con 3.746 víctimas y 106 muertos.
- Las cámaras están en el 7,7 % de los cuadros con víctimas, y ahí pasa el 13,9 % de las víctimas. O sea, están en zonas con muchas víctimas, pero son pocas.

En todas las pruebas más de la mitad de los puntos críticos sigue sin cámara, así que la conclusión no cambia.